In [4]:
import requests
import pandas as pd
from IPython.display import display

# CMS Open Payments - 2024 General Payments
url = (
    "https://openpaymentsdata.cms.gov/api/1/datastore/query/"
    "e6b17c6a-2534-4207-a4a1-6746a14911ff/0"
)

# Retrieve 1,000 records
response = requests.get(
    url,
    params={"limit": 500},
    timeout=120
)

# Display API error details if the request fails
if not response.ok:
    print("Status:", response.status_code)
    print("Error:", response.text[:1000])

response.raise_for_status()

data = response.json()
payments = pd.DataFrame(data["results"])

# Dataset information
print("=" * 50)
print("CMS OPEN PAYMENTS - 2024")
print("=" * 50)

print("Rows:", len(payments))
print("Columns:", len(payments.columns))

print("\nFIRST 5 ROWS:")
display(payments.head())

print("\nSCHEMA:")
payments.info()

print("\nCOLUMN NAMES:")
print(payments.columns.tolist())

# NPI check
print("\nNPI CHECK:")

npi_col = next(
    (col for col in payments.columns
     if col.lower() == "covered_recipient_npi"),
    None
)

if npi_col:
    npi = payments[npi_col].astype("string").str.strip()
    valid = npi.str.fullmatch(r"\d{10}").fillna(False)

    print("NPI column:", npi_col)
    print("Valid NPIs:", valid.sum())
    print("Unique valid NPIs:", npi[valid].nunique())

    display(payments.loc[valid, [npi_col]].head(10))
else:
    print("NPI column not found")

CMS OPEN PAYMENTS - 2024
Rows: 500
Columns: 91

FIRST 5 ROWS:


,change_type,covered_recipient_type,teaching_hospital_ccn,teaching_hospital_id,teaching_hospital_name,covered_recipient_profile_id,covered_recipient_npi,covered_recipient_first_name,covered_recipient_middle_name,covered_recipient_last_name,...,associated_drug_or_biological_ndc_4,associated_device_or_medical_supply_pdi_4,covered_or_noncovered_indicator_5,indicate_drug_or_biological_or_device_or_medical_supply_5,product_category_or_therapeutic_area_5,name_of_drug_or_biological_or_device_or_medical_supply_5,associated_drug_or_biological_ndc_5,associated_device_or_medical_supply_pdi_5,program_year,payment_publication_date
0,UNCHANGED,Covered Recipient Non-Physician Practitioner,,,,10593686,1790251015,AMY,,PONZIO,...,,,,,,,,,2024,06/30/2026
1,UNCHANGED,Covered Recipient Physician,,,,232750,1134115629,CHRISTOPHER,,STALLTER,...,,,,,,,,,2024,06/30/2026
2,UNCHANGED,Covered Recipient Physician,,,,9569889,1417453861,ANDREA,,TAI,...,,,,,,,,,2024,06/30/2026
3,UNCHANGED,Covered Recipient Non-Physician Practitioner,,,,11109983,1073086989,ANGELA,,NEWMAN,...,,,,,,,,,2024,06/30/2026
4,UNCHANGED,Covered Recipient Physician,,,,4432051,1942795463,ANNA,,MCFARLAND,...,,,,,,,,,2024,06/30/2026



SCHEMA:
<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 91 columns):
 #   Column                                                            Non-Null Count  Dtype
---  ------                                                            --------------  -----
 0   change_type                                                       500 non-null    str  
 1   covered_recipient_type                                            500 non-null    str  
 2   teaching_hospital_ccn                                             500 non-null    str  
 3   teaching_hospital_id                                              500 non-null    str  
 4   teaching_hospital_name                                            500 non-null    str  
 5   covered_recipient_profile_id                                      500 non-null    str  
 6   covered_recipient_npi                                             500 non-null    str  
 7   covered_recipient_first_name                           

,covered_recipient_npi
0,1790251015
1,1134115629
2,1417453861
3,1073086989
4,1942795463
5,1811362197
6,1699989384
7,1225352792
8,1417957135
9,1497000590


In [6]:
from pathlib import Path

Path("../data/processed").mkdir(parents=True, exist_ok=True)

payments.to_csv("../data/processed/payments_sample.csv", index=False)